# Experiment 4 Postprocessing: Scalability With Unison ON/OFF

This notebook compares scalability results for both modes per client count:
- `unison_off`
- `unison_on`

Main output: common comparison tables/figures showing the effect of Unison.


In [5]:
from __future__ import annotations

from pathlib import Path
import csv
import json
import re
from datetime import datetime
from typing import Any

import matplotlib.pyplot as plt

plt.rcParams.update({
    "figure.figsize": (6, 4),      # width, height in inches
    "font.size": 14,               # base font size
    "axes.titlesize": 16,
    "axes.labelsize": 15,
    "xtick.labelsize": 13,
    "ytick.labelsize": 13,
    "legend.fontsize": 13,
    "figure.dpi": 300,             # high resolution for paper
})

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / 'orchestrator.py').exists() and (candidate / 'experiments').is_dir():
            return candidate
    return None


ROOT = find_repo_root(Path.cwd())
if ROOT is None:
    raise RuntimeError(f'Could not locate fl-net-sim repository root from {Path.cwd()}')

EXP_ID = 'exp4_scalability_clients'
EXP_DIR = ROOT / 'experiments' / EXP_ID
OUTPUTS_DIR = EXP_DIR / 'outputs'
SOURCE_EXPORTS_DIR = OUTPUTS_DIR / 'source_exports'
TABLES_DIR = OUTPUTS_DIR / 'tables'
FIGURES_DIR = OUTPUTS_DIR / 'figures'
MANIFEST_PATH = OUTPUTS_DIR / 'last_run_manifest.json'

TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODE_ORDER = ['off', 'on']
MODE_LABEL = {'off': 'Unison Off', 'on': 'Unison On'}
ID_RE = re.compile(r'^wifi_c(?P<clients>\d{3})_ap(?P<aps>\d{2})_unison_(?P<mode>on|off)$')

print(f'ROOT={ROOT}')
print(f'EXP_DIR={EXP_DIR}')


ROOT=/mnt/nas_drive/psklavos/MDM2026/fl-net-sim
EXP_DIR=/mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp4_scalability_clients


In [6]:
def to_markdown_no_deps(df: pd.DataFrame) -> str:
    try:
        return df.to_markdown(index=False)
    except Exception:
        safe = df.fillna('')
        cols = [str(c) for c in safe.columns]
        header = '| ' + ' | '.join(cols) + ' |'
        sep = '| ' + ' | '.join(['---'] * len(cols)) + ' |'
        rows = [
            '| ' + ' | '.join(str(v).replace('\n', ' ').replace('|', '\\|') for v in row) + ' |'
            for row in safe.itertuples(index=False, name=None)
        ]
        return '\n'.join([header, sep, *rows])


def load_round_table(path: Path) -> pd.DataFrame:
    if not path.exists():
        return pd.DataFrame()
    text = path.read_text(encoding='utf-8', errors='ignore')
    first_section = text.split('\n\n', 1)[0].strip()
    if not first_section:
        return pd.DataFrame()
    rows = list(csv.DictReader(first_section.splitlines()))
    frame = pd.DataFrame(rows)
    if frame.empty:
        return frame
    for col in frame.columns:
        converted = pd.to_numeric(frame[col], errors='coerce')
        if converted.notna().any():
            frame[col] = converted
    return frame


NETWORK_RE = re.compile(
    r'network_type=\w+\s+rounds=(?P<rounds>\d+)\s+parallelism=(?P<parallelism>\d+)\s+cached=(?P<cached>\d+)\s+deduped=(?P<deduped>\d+)'
)
PLANNED_RE = re.compile(r'planned\s+(?P<planned>\d+)\s+rounds:')
TS_RE = re.compile(r'^(?P<ts>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}(?:,\d+)?)\s+\|\s+INFO\s+\|')
TS_FORMATS = ['%Y-%m-%d %H:%M:%S', '%Y-%m-%d %H:%M:%S,%f']


def parse_log_metrics(log_path: Path) -> dict[str, Any]:
    out = {
        'rounds_requested': 0,
        'planned_rounds': 0,
        'cached_rounds': 0,
        'deduped_rounds': 0,
        'parallelism_reported': 0,
        'exec_time_from_log_s': np.nan,
    }
    if not log_path.exists():
        return out

    text = log_path.read_text(encoding='utf-8', errors='ignore')

    network = NETWORK_RE.findall(text)
    if network:
        rounds, parallelism, cached, deduped = network[-1]
        out['rounds_requested'] = int(rounds)
        out['parallelism_reported'] = int(parallelism)
        out['cached_rounds'] = int(cached)
        out['deduped_rounds'] = int(deduped)

    planned = PLANNED_RE.findall(text)
    if planned:
        out['planned_rounds'] = int(planned[-1])

    first_ts = None
    last_ts = None
    for line in text.splitlines():
        m = TS_RE.match(line)
        if not m:
            continue
        ts_text = m.group('ts')
        ts = None
        for fmt in TS_FORMATS:
            try:
                ts = datetime.strptime(ts_text, fmt)
                break
            except ValueError:
                continue
        if ts is None:
            continue
        if first_ts is None:
            first_ts = ts
        last_ts = ts

    if first_ts is not None and last_ts is not None and last_ts >= first_ts:
        out['exec_time_from_log_s'] = (last_ts - first_ts).total_seconds()

    return out


def parse_config_id(config_id: str) -> tuple[int, int, str] | None:
    m = ID_RE.match(config_id)
    if not m:
        return None
    return int(m.group('clients')), int(m.group('aps')), m.group('mode')


In [7]:
manifest_rows: dict[str, dict[str, Any]] = {}
if MANIFEST_PATH.exists():
    try:
        payload = json.loads(MANIFEST_PATH.read_text(encoding='utf-8'))
        for row in payload.get('runs', []):
            config_id = str(row.get('config_id', '')).strip()
            if config_id:
                manifest_rows[config_id] = row
        print(f'Loaded manifest metadata from {MANIFEST_PATH} (runs={len(manifest_rows)})')
    except Exception as exc:
        print(f'Warning: failed to parse manifest: {exc}')
else:
    print(f'No manifest found at {MANIFEST_PATH}. Folder-only mode.')

if not SOURCE_EXPORTS_DIR.exists():
    raise FileNotFoundError(f'Missing source exports directory: {SOURCE_EXPORTS_DIR}')

source_config_ids = sorted(
    p.name for p in SOURCE_EXPORTS_DIR.iterdir()
    if p.is_dir() and parse_config_id(p.name) is not None
)
if not source_config_ids:
    source_config_ids = sorted(
        cid for cid in manifest_rows.keys()
        if parse_config_id(cid) is not None
    )

if not source_config_ids:
    raise ValueError(
        'No exp4 run outputs found. Run: bash experiments/exp4_scalability_clients/run_experiment.sh'
    )

summary_rows: list[dict[str, Any]] = []
per_round_parts: list[pd.DataFrame] = []

for config_id in source_config_ids:
    parsed_id = parse_config_id(config_id)
    if parsed_id is None:
        continue
    clients, aps, mode = parsed_id

    exports_dir = SOURCE_EXPORTS_DIR / config_id
    row_meta = manifest_rows.get(config_id, {})

    requested_csv = exports_dir / 'requested_rounds.csv'
    if not requested_csv.exists() and row_meta.get('copied_requested_rounds_csv'):
        requested_csv = Path(row_meta['copied_requested_rounds_csv'])

    log_path = exports_dir / 'orchestrator_capture.log'
    if not log_path.exists() and row_meta.get('capture_log'):
        log_path = Path(row_meta['capture_log'])

    if not requested_csv.exists() and not log_path.exists():
        continue

    req = load_round_table(requested_csv)
    parsed = parse_log_metrics(log_path)

    rounds_requested = int(parsed['rounds_requested'])
    planned_rounds = int(parsed['planned_rounds'])
    cached_rounds = int(parsed['cached_rounds'])
    deduped_rounds = int(parsed['deduped_rounds'])
    parallelism_reported = int(parsed['parallelism_reported'])

    if rounds_requested == 0 and not req.empty:
        rounds_requested = int(len(req))
    if planned_rounds == 0:
        planned_rounds = rounds_requested

    exec_time_s = row_meta.get('exec_time_s', np.nan)
    exec_time_s = pd.to_numeric(pd.Series([exec_time_s]), errors='coerce').iloc[0]
    if pd.isna(exec_time_s):
        exec_time_s = parsed['exec_time_from_log_s']

    mean_comm_total = float(req['comm_total_s'].mean()) if 'comm_total_s' in req.columns and not req.empty else np.nan
    p95_comm_total = float(req['comm_total_s'].quantile(0.95)) if 'comm_total_s' in req.columns and not req.empty else np.nan
    mean_dl = float(req['agg_dl_goodput_mbps'].mean()) if 'agg_dl_goodput_mbps' in req.columns and not req.empty else np.nan
    mean_ul = float(req['agg_ul_goodput_mbps'].mean()) if 'agg_ul_goodput_mbps' in req.columns and not req.empty else np.nan

    summary_rows.append(
        {
            'config_id': config_id,
            'clients': int(clients),
            'ap_count': int(aps),
            'unison_mode': mode,
            'exec_time_s': float(exec_time_s) if pd.notna(exec_time_s) else np.nan,
            'rounds_requested': rounds_requested,
            'planned_rounds': planned_rounds,
            'cached_rounds': cached_rounds,
            'deduped_rounds': deduped_rounds,
            'parallelism_reported': parallelism_reported,
            'mean_comm_total_s': mean_comm_total,
            'p95_comm_total_s': p95_comm_total,
            'mean_agg_dl_goodput_mbps': mean_dl,
            'mean_agg_ul_goodput_mbps': mean_ul,
            'requested_rounds_csv': str(requested_csv) if requested_csv.exists() else '',
            'orchestrator_capture_log': str(log_path) if log_path.exists() else '',
        }
    )

    if not req.empty:
        per = req.copy()
        per['config_id'] = config_id
        per['clients'] = int(clients)
        per['ap_count'] = int(aps)
        per['unison_mode'] = mode
        per_round_parts.append(per)

summary = pd.DataFrame(summary_rows)
if summary.empty:
    raise ValueError('No run outputs found under outputs/source_exports for exp4.')

summary['unison_mode'] = pd.Categorical(summary['unison_mode'], categories=MODE_ORDER, ordered=True)
summary = summary.sort_values(['clients', 'unison_mode']).reset_index(drop=True)
summary['unison_mode'] = summary['unison_mode'].astype(str)
summary['unison_label'] = summary['unison_mode'].map(MODE_LABEL)
summary['avg_exec_time_per_round_s'] = summary['exec_time_s'] / summary['rounds_requested'].clip(lower=1)

summary_out = summary[
    [
        'config_id',
        'clients',
        'ap_count',
        'unison_mode',
        'exec_time_s',
        'avg_exec_time_per_round_s',
        'rounds_requested',
        'mean_comm_total_s',
        'p95_comm_total_s',
        'mean_agg_dl_goodput_mbps',
        'mean_agg_ul_goodput_mbps',
    ]
].copy()

summary_csv = TABLES_DIR / 'exp4_scalability_summary.csv'
summary_md = TABLES_DIR / 'exp4_scalability_summary.md'
summary_out.to_csv(summary_csv, index=False)
summary_md.write_text(to_markdown_no_deps(summary_out), encoding='utf-8')

pivot_exec = summary.pivot_table(index=['clients', 'ap_count'], columns='unison_mode', values='exec_time_s', aggfunc='first')
pivot_round = summary.pivot_table(index=['clients', 'ap_count'], columns='unison_mode', values='avg_exec_time_per_round_s', aggfunc='first')
pivot_comm = summary.pivot_table(index=['clients', 'ap_count'], columns='unison_mode', values='mean_comm_total_s', aggfunc='first')

comparison = pivot_exec.reset_index()[['clients', 'ap_count']].copy()
index_ref = pivot_exec.index

def values_or_nan(pivot: pd.DataFrame, col: str, index_ref: pd.Index) -> np.ndarray:
    if col in pivot.columns:
        return pivot.reindex(index_ref)[col].to_numpy()
    return np.full(len(index_ref), np.nan)

comparison['exec_time_unison_off_s'] = values_or_nan(pivot_exec, 'off', index_ref)
comparison['exec_time_unison_on_s'] = values_or_nan(pivot_exec, 'on', index_ref)
comparison['avg_round_time_unison_off_s'] = values_or_nan(pivot_round, 'off', index_ref)
comparison['avg_round_time_unison_on_s'] = values_or_nan(pivot_round, 'on', index_ref)
comparison['mean_comm_total_unison_off_s'] = values_or_nan(pivot_comm, 'off', index_ref)
comparison['mean_comm_total_unison_on_s'] = values_or_nan(pivot_comm, 'on', index_ref)

comparison['exec_time_speedup_on_vs_off'] = comparison['exec_time_unison_off_s'] / comparison['exec_time_unison_on_s']
comparison['avg_round_speedup_on_vs_off'] = comparison['avg_round_time_unison_off_s'] / comparison['avg_round_time_unison_on_s']
comparison['comm_speedup_on_vs_off'] = comparison['mean_comm_total_unison_off_s'] / comparison['mean_comm_total_unison_on_s']
comparison = comparison.sort_values('clients').reset_index(drop=True)

comp_csv = TABLES_DIR / 'exp4_unison_comparison.csv'
comp_md = TABLES_DIR / 'exp4_unison_comparison.md'
comparison.to_csv(comp_csv, index=False)
comp_md.write_text(to_markdown_no_deps(comparison), encoding='utf-8')

if per_round_parts:
    per_round = pd.concat(per_round_parts, ignore_index=True)
    first_cols = ['config_id', 'clients', 'ap_count', 'unison_mode']
    rest = [c for c in per_round.columns if c not in first_cols]
    per_round = per_round[first_cols + rest]
else:
    per_round = pd.DataFrame(columns=['config_id', 'clients', 'ap_count', 'unison_mode'])

per_round_csv = TABLES_DIR / 'exp4_scalability_per_round.csv'
per_round.to_csv(per_round_csv, index=False)

print(f'Wrote {summary_csv}')
print(f'Wrote {summary_md}')
print(f'Wrote {comp_csv}')
print(f'Wrote {comp_md}')
print(f'Wrote {per_round_csv}')
comparison


Loaded manifest metadata from /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp4_scalability_clients/outputs/last_run_manifest.json (runs=16)
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp4_scalability_clients/outputs/tables/exp4_scalability_summary.csv
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp4_scalability_clients/outputs/tables/exp4_scalability_summary.md
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp4_scalability_clients/outputs/tables/exp4_unison_comparison.csv
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp4_scalability_clients/outputs/tables/exp4_unison_comparison.md
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp4_scalability_clients/outputs/tables/exp4_scalability_per_round.csv


unison_mode,clients,ap_count,exec_time_unison_off_s,exec_time_unison_on_s,avg_round_time_unison_off_s,avg_round_time_unison_on_s,mean_comm_total_unison_off_s,mean_comm_total_unison_on_s,exec_time_speedup_on_vs_off,avg_round_speedup_on_vs_off,comm_speedup_on_vs_off
0,1,1,2.587427,3.078551,2.587427,3.078551,1.832509,1.832509,0.840469,0.840469,1.0
1,2,1,4.577591,4.160585,4.577591,4.160585,2.466981,2.466981,1.100228,1.100228,1.0
2,4,2,8.659648,7.223020,8.659648,7.223020,4.649380,4.649380,1.198896,1.198896,1.0
3,8,3,18.891232,12.121375,18.891232,12.121375,5.385138,5.385138,1.558506,1.558506,1.0
4,16,4,40.768142,25.846574,40.768142,25.846574,8.043149,8.043149,1.577313,1.577313,1.0
5,32,5,159.160953,93.441015,159.160953,93.441015,11.945611,11.945611,1.703331,1.703331,1.0
6,64,6,444.025801,134.373862,444.025801,134.373862,21.586567,21.586567,3.304406,3.304406,1.0
7,128,7,1221.607098,367.003256,1221.607098,367.003256,NaN,55.355740,3.328600,3.328600,NaN


In [8]:
fig1 = FIGURES_DIR / 'exp4_exec_time_vs_clients_by_unison.png'
fig2 = FIGURES_DIR / 'exp4_avg_round_time_vs_clients_by_unison.png'
fig3 = FIGURES_DIR / 'exp4_unison_speedup_vs_clients.png'

plt.figure(figsize=(9, 5))
for mode in MODE_ORDER:
    group = summary[summary['unison_mode'] == mode].copy()
    if group.empty:
        continue
    x = group['clients'].to_numpy()
    y = group['exec_time_s'].to_numpy()
    plt.plot(x, y, marker='o', linewidth=2, label=MODE_LABEL.get(mode, mode))
plt.xscale('log', base=2)
xticks = sorted(summary['clients'].unique())
plt.xticks(xticks, [str(v) for v in xticks])
plt.xlabel('Selected Clients (all selected each round)')
plt.ylabel('Total Execution Time (s)')
plt.title('Total Execution Time vs Clients (Unison ON/OFF)')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(fig1, dpi=180)
plt.close()

plt.figure(figsize=(9, 5))
for mode in MODE_ORDER:
    group = summary[summary['unison_mode'] == mode].copy()
    if group.empty:
        continue
    x = group['clients'].to_numpy()
    y = group['avg_exec_time_per_round_s'].to_numpy()
    plt.plot(x, y, marker='o', linewidth=2, label=MODE_LABEL.get(mode, mode))
plt.xscale('log', base=2)
plt.xticks(xticks, [str(v) for v in xticks])
plt.xlabel('Selected Clients')
plt.ylabel('Average Execution Time per Round (s)')
plt.title('Avg Round Time vs Clients (Unison ON/OFF)')
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.savefig(fig2, dpi=180)
plt.close()

plt.figure(figsize=(9, 5))
plt.plot(comparison['clients'], comparison['exec_time_speedup_on_vs_off'], marker='o', linewidth=2)
plt.xscale('log', base=2)
plt.xticks(comparison['clients'], [str(v) for v in comparison['clients']])
plt.xlabel('Selected Clients')
plt.ylabel('Speedup (Off / On)')
plt.title('Unison Speedup Against Baseline vs Clients')
plt.grid(alpha=0.25)
plt.tight_layout()
plt.savefig(fig3, dpi=180)
plt.close()

print(f'Wrote {fig1}')
print(f'Wrote {fig2}')
print(f'Wrote {fig3}')


Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp4_scalability_clients/outputs/figures/exp4_exec_time_vs_clients_by_unison.png
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp4_scalability_clients/outputs/figures/exp4_avg_round_time_vs_clients_by_unison.png
Wrote /mnt/nas_drive/psklavos/MDM2026/fl-net-sim/experiments/exp4_scalability_clients/outputs/figures/exp4_unison_speedup_vs_clients.png
